In [1]:
import pandas as pd
from statsbombpy import sb

In [2]:
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 40)

In [3]:
competitions = sb.competitions()
print(f"Found {len(competitions)} competition-season combinations.\n")
 
table = competitions[
    ["competition_id", "season_id", "country_name", "competition_name", "season_name"]
].sort_values(["competition_name", "season_name"])
print(table.to_string(index=False))

/Users/alper/anaconda3/envs/football/lib/python3.11/site-packages/statsbombpy/api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Found 80 competition-season combinations.

 competition_id  season_id              country_name        competition_name season_name
              9         27                   Germany           1. Bundesliga   2015/2016
              9        281                   Germany           1. Bundesliga   2023/2024
           1267        107                    Africa  African Cup of Nations        2023
             16        276                    Europe        Champions League   1970/1971
             16         71                    Europe        Champions League   1971/1972
             16        277                    Europe        Champions League   1972/1973
             16         76                    Europe        Champions League   1999/2000
             16         44                    Europe        Champions League   2003/2004
             16         37                    Europe        Champions League   2004/2005
             16         39                    Europe        Champio

In [4]:
print("\n--- Available seasons per competition ---")
print(competitions.groupby("competition_name")["season_name"].apply(list).to_string())


--- Available seasons per competition ---
competition_name
1. Bundesliga                               [2023/2024, 2015/2016]
African Cup of Nations                                      [2023]
Champions League           [2018/2019, 2017/2018, 2016/2017, 20...
Copa America                                                [2024]
Copa del Rey                     [1983/1984, 1982/1983, 1977/1978]
FA Women's Super League    [2023/2024, 2020/2021, 2019/2020, 20...
FIFA U20 World Cup                                          [1979]
FIFA World Cup             [2022, 2018, 1990, 1986, 1974, 1970,...
Frauen Bundesliga                                      [2023/2024]
Indian Super league                                    [2021/2022]
La Liga                    [2020/2021, 2019/2020, 2018/2019, 20...
Liga F                                                 [2023/2024]
Liga Profesional                                 [1997/1998, 1981]
Ligue 1                          [2022/2023, 2021/2022, 2015/2016]
Ma

In [5]:
COMP_ID = 2
SEASON_ID = 27
 
if COMP_ID is None or SEASON_ID is None:
    import sys
    print(
        "\nCOMP_ID / SEASON_ID not set yet. Check the table above, pick a "
        "competition_id/season_id, fill them in, and re-run."
    )
    sys.exit(0)
 

In [6]:
# Match list for the selected competition/season
matches = sb.matches(competition_id=COMP_ID, season_id=SEASON_ID)
matches[
    ["match_id", "match_date", "home_team", "away_team", "home_score", "away_score"]
]

/Users/alper/anaconda3/envs/football/lib/python3.11/site-packages/statsbombpy/api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,match_id,match_date,home_team,away_team,home_score,away_score
0,3754217,2015-09-19,Chelsea,Arsenal,2,0
1,3754117,2015-12-13,Aston Villa,Arsenal,0,2
2,3754296,2015-12-21,Arsenal,Manchester City,2,1
3,3753983,2015-10-31,Swansea City,Arsenal,0,3
4,3754160,2015-12-05,Arsenal,Sunderland,3,1
...,...,...,...,...,...,...
375,3754020,2015-08-17,Liverpool,AFC Bournemouth,1,0
376,3754267,2015-08-15,Watford,West Bromwich Albion,0,0
377,3754141,2015-08-09,Arsenal,West Ham United,0,2
378,3754128,2015-08-08,AFC Bournemouth,Aston Villa,0,1


In [7]:
# Fetch one match's event data and inspect its general structure
sample_match_id = matches.iloc[0]["match_id"]
events = sb.events(match_id=sample_match_id)

print(f"Match: {matches.iloc[0]['home_team']} vs {matches.iloc[0]['away_team']}")
print(f"Total event count: {len(events)}\n")
print(events["type"].value_counts())

/Users/alper/anaconda3/envs/football/lib/python3.11/site-packages/statsbombpy/api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Match: Chelsea vs Arsenal
Total event count: 3732

type
Pass                 1046
Ball Receipt*         963
Carry                 814
Pressure              320
Ball Recovery         106
Duel                   78
Dribble                52
Clearance              42
Block                  41
Goal Keeper            34
Dribbled Past          33
Shot                   33
Foul Committed         32
Foul Won               29
Miscontrol             27
Dispossessed           26
Interception           20
Substitution            6
Half Start              4
Half End                4
Injury Stoppage         4
Tactical Shift          3
Shield                  3
Bad Behaviour           3
Starting XI             2
Referee Ball-Drop       2
Player Off              1
Player On               1
Error                   1
Own Goal For            1
Own Goal Against        1
Name: count, dtype: int64


In [8]:
# Which columns are populated for each event type - useful for seeing
# what information will be available to the agent/reporting layer
for event_type in events["type"].unique():
    sample = events[events["type"] == event_type].iloc[0]
    non_null_cols = sample.dropna().index.tolist()
    print(f"\n{event_type}: {non_null_cols}")


Starting XI: ['duration', 'id', 'index', 'match_id', 'minute', 'period', 'play_pattern', 'possession', 'possession_team', 'possession_team_id', 'second', 'tactics', 'team', 'team_id', 'timestamp', 'type']

Half Start: ['duration', 'id', 'index', 'match_id', 'minute', 'period', 'play_pattern', 'possession', 'possession_team', 'possession_team_id', 'related_events', 'second', 'team', 'team_id', 'timestamp', 'type']

Pass: ['duration', 'id', 'index', 'location', 'match_id', 'minute', 'pass_angle', 'pass_body_part', 'pass_end_location', 'pass_height', 'pass_length', 'pass_recipient', 'pass_recipient_id', 'pass_type', 'period', 'play_pattern', 'player', 'player_id', 'position', 'possession', 'possession_team', 'possession_team_id', 'related_events', 'second', 'team', 'team_id', 'timestamp', 'type']

Ball Receipt*: ['id', 'index', 'location', 'match_id', 'minute', 'period', 'play_pattern', 'player', 'player_id', 'position', 'possession', 'possession_team', 'possession_team_id', 'related_eve